## Application Programming Interfaces and Internal Functions

Libraries are useful for researchers and developers, i.e. for people that are working with code themselves. Some functions in the library are meant to be *exposed*, while other *internal* functions, such as *helper functions*, are meant to remain hidden from people using the library. The collection of functions and variables that is exposed is called the *application programming interface*, or *API*. The main purposes of APIs are 1) to hide the internal parts of how a system works, exposing only those parts a researcher or developer will find useful, and 2) keeping the exposed parts consistent, even if the internal details later change.

In Python, the internal functions and variables that have no use outside the current namespace, such as helper functions, should be prefixed by an underscore (`_`). When a module is imported into the current namespace, all these internal functions and variables remain hidden. This keeps the exposed API clean and allows the library developers to change the internal details without changing the API:

In [3]:
# An internal helper function. This function is not exposed in the API.
def _get_z_position(dicom_image):
    """
    Returns the z-coordinate of the Image Position Patient DICOM attribute.

    :param dicom_image: A pydicom dataset
    :returns: The z-coordinate
    """
    return dicom_image.ImagePositionPatient[2]


# A public function. This function is exposed in the API.
def read_dicom(file_path):
    """
    Reads a DICOM file and returns the pixel data.

    :param file_path: The path to the DICOM file.
    :returns: A numpy array with pixel data.
    """
    dicom_image = dcmread(file_path)  # Read the image.
    return apply_rescale(dicom_image.pixel_array, dicom_image)  # Convert from stored pixel values to real values

## Commandline Interfaces

To make software usable by a wider audience, i.e. for people that don't have any (Python) programming experience, we should add a *user interface*. The most basic form of such a user interface is the *command line interface*, or CLI. A CLI is simply a script or *executable* that is meant to be called from the *command line* (e.g. the PowerShell terminal in Windows or the Terminal app on macOS). Note that *command line*, *terminal* and *shell* are synonyms in this context. The code that is executed when someone calls this executable is called the *entrypoint*.

An example of such a command line application is `python` itself. Jupyter allows calling command line executables using the exclamation mark (`!`), so we can demo it here:

In [9]:
!python --help

usage: python [option] ... [-c cmd | -m mod | file | -] [arg] ...
Options (and corresponding environment variables):
-b     : issue warnings about converting bytes/bytearray to str and comparing
         bytes/bytearray with str or bytes with int. (-bb: issue errors)
-B     : don't write .pyc files on import; also PYTHONDONTWRITEBYTECODE=x
-c cmd : program passed in as string (terminates option list)
-d     : turn on parser debugging output (for experts only, only works on
         debug builds); also PYTHONDEBUG=x
-E     : ignore PYTHON* environment variables (such as PYTHONPATH)
-h     : print this help message and exit (also -? or --help)
-i     : inspect interactively after running script; forces a prompt even
         if stdin does not appear to be a terminal; also PYTHONINSPECT=x
-I     : isolate Python from the user's environment (implies -E, -P and -s)
-m mod : run library module as a script (terminates option list)
-O     : remove assert and __debug__-dependent statements; add

As you can see in the output of the `python --help` command, a CLI can also have options and arguments. The option `--help` prints the help. The first argument to `python` is the script to execute, so `python my_module.py` executes the code in the file `my_module.py`.

It is common practice to check if a module is executed as the *main* script before executing the desired entrypoint function. To do so, we need to check the value of the `__name__` variable, which Python automatically sets in the module's global namespace. The value for `__name__` is `"__main__"` when the module is executed as the main script:

```python
# This is our entrypoint function
def hello():
    print("Hello World!")


# Execute the entrypoint function if this module is executed as the main script.
if __name__ == "__main__":
    hello()
```

### Click

Click is a Python package for creating command line interfaces with as little code as necessary. It uses *decorator functions* to provide a Python function with the options, and arguments passed on the command line. A decorator function starts with a `@`, and wraps another function with extra behavior without changing the function itself. You don't have to fully understand how decorators work, but if you're interested you can read more [here](https://www.geeksforgeeks.org/python/decorators-in-python/). A simple Click command line interface looks like this:

```python
import click


@click.command()
@click.option('--count', default=1, help='Number of greetings.')
@click.argument('name', help='The person to greet.')
def hello(count, name):
    """Simple program that greets NAME for a total of COUNT times."""
    for x in range(count):
        click.echo(f"Hello {name}!")


if __name__ == '__main__':
    hello()
```

This is what it looks like if this command like application is executed:

```shell
python hello.py --count=3 Brian
Hello Brian!
Hello Brian!
Hello Brian!
```

As a bonus, Click automatically generates nicely formatted help pages:
```shell
python hello.py --help
Usage: hello.py [OPTIONS] NAME

  Simple program that greets NAME for a total of COUNT times.

Positional arguments:
  NAME  The person to greet.

Options:
  --count INTEGER  Number of greetings.
  --help           Show this message and exit.
```



## Exception Handling

Exception Handling allows a program to handle unexpected errors during execution in a controlled way, instead of crashing abruptly. It enables programs to detect errors, manage them properly and continue execution wherever possible.

For example, think of a command line application for DICOM file conversion, that searches and converts all DICOM files in nested directories to Nifti. You probably don't want the application to stop when `pydicom.dcmread()` *raises* the *exception* `pydicom.errors.InvalidDicomError` when it encountered another filetype. Instead, you would want to print a warning message instead and continue with the next file.

This can be achieved by using the `try` and `except` keywords. Python tries to execute the code in a `try` block, and when it encounters a specific type of exception, the body of the `except` block is executed:

In [3]:
from pydicom import dcmread
from pydicom.errors import InvalidDicomError


file_path = 'images/python-logo-pmi.png'  # This is not a DICOM file
try:
    dicom_image = dcmread(file_path)  # This will raise an InvalidDicomError
except InvalidDicomError:  # This 'catches' an InvalidDicomError
    print(f'Could not read: {file_path}')  # This is executed when an InvalidDicomError is caught

Could not read: images/python-logo-pmi.png


`try` and `except` can be extended with `else` and `finally` to execute code when no errors occur, or to execute code regardless of what happens.

> **External course material**
>
> The keywords `try`, `except`, `else`, and `finally` are very well explained on the webpage below. Review this webpage:
>
> * <https://www.geeksforgeeks.org/python/python-exception-handling/>